# GPT from Scratch — Colab Training

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

**Stages:**
1. Setup (clone repo, install deps)
2. Download datasets
3. Pretrain on WikiText-103 (~45 min on T4)
4. Fine-tune on DailyDialog (~15 min on T4)
5. Download checkpoint to your machine

## 1. Setup

In [ ]:
# Check GPU
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clone your repo — replace with your actual GitHub URL
GITHUB_URL = 'https://github.com/DudeAj/gpt-from-scratch.git'

import os
# Colab clones into /content/gpt-from-scratch/
# We cd into it so all python -m commands resolve correctly
!git clone {GITHUB_URL}
os.chdir('/content/gpt-from-scratch')
!pwd
!git log --oneline -5

In [ ]:
# Install dependencies
!pip install datasets tokenizers tqdm -q

# Verify torch version
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}')

## 2. Download datasets

In [ ]:
# Download WikiText-103 + DailyDialog (~500 MB, takes ~3-5 min)
!python -m dataset.download_data

## 3. Pretrain on WikiText-103

Expected time on T4: **~45 minutes**  
First run also tokenises WikiText-103 (adds ~5 min, cached after that)

Watch the loss — it should drop from ~10 to ~4 over 3 epochs.

In [ ]:
!python -m training.pretrain

## 4. Fine-tune on DailyDialog

Expected time on T4: **~15 minutes**  
Loss is masked (assistant tokens only) so numbers will be lower than pretrain.

In [ ]:
!python -m training.finetune

## 5. Quick generation test

In [ ]:
import torch
from model.gpt import GPT
from dataset.dialog_dataset import load_dialog_tokenizer
from inference.chat import generate_reply

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load('checkpoints/finetune_best.pt', map_location=device)
cfg  = ckpt['config']

tokenizer, _ = load_dialog_tokenizer()

model = GPT(
    vocab_size  = cfg['vocab_size'],
    max_seq_len = cfg['max_seq_len'],
    d_model     = cfg['d_model'],
    num_heads   = cfg['num_heads'],
    num_layers  = cfg['num_layers'],
    dropout     = cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model'])

# Test a few prompts
prompts = [
    'Hello, how are you?',
    'What do you like to do on weekends?',
    'Can you recommend a good book?',
]

for p in prompts:
    reply = generate_reply(model, tokenizer, [p], temperature=0.8, top_k=40)
    print(f'You : {p}')
    print(f'Bot : {reply}')
    print()

## 6. Download checkpoints to your local machine

In [ ]:
# Zip and download both checkpoints
!zip -r checkpoints.zip checkpoints/

from google.colab import files
files.download('checkpoints.zip')

## (Optional) Save to Google Drive instead

If your session disconnects before the download finishes, save to Drive first:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    'checkpoints',
    '/content/drive/MyDrive/gpt-checkpoints',
    dirs_exist_ok=True
)
print('Saved to Google Drive!')